[Guide Index](README.md)

# 04.04 SRL vs. SPARQL

SRL closely follows SPARQL, with some notable differences. This guide compares an SRL rule with a SPARQL `CONSTRUCT` and the `WHERE` clause of a rule with SPARQL's, and shows what a rule does in place of what SRL leaves out.

[04.01](04.01-srl-sparql-rules.ipynb) shows how to write rules, [04.02](04.02-srl-running-rule-sets.ipynb) explains how a rule set runs, and [04.03](04.03-srl-more-features.ipynb) covers more features.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [2]:
from starlayer.graph import NamedNode, StarLayerDataset, StarLayerGraph, isomorphic
from starlayer.sparql import srl

EX = "http://example.org/"


def ex(name):
    """A term in the example namespace, as a pyoxigraph NamedNode."""
    return NamedNode(EX + name)


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses.
    (A new StarLayerGraph has no prefixes; the ones used here are bound explicitly.)"""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 04.04.1 The base graph

The examples use a small software-dependency graph, the same running example as [the SRL spec's worked example](https://www.w3.org/TR/sparql12-rl/#worked-example): a `:frontend` depends on an `:app`, which depends on a `:db` that has a known vulnerability. A `:logger` component has no vulnerabilities at all.

In [3]:
base = StarLayerGraph(prefixes={"ex": EX})
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .  # no vulnerability

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")


print("data graph")
show(base)

data graph
@prefix ex: <http://example.org/> .

ex:app a ex:Component ;
    ex:dependsOn ex:db .

ex:db a ex:Component ;
    ex:hasVulnerability ex:vuln1 .

ex:frontend a ex:Component ;
    ex:dependsOn ex:app .

ex:logger a ex:Component .

ex:vuln1 ex:severity 9.1 .



## 04.04.2 An SRL rule is like SPARQL CONSTRUCT

An SRL rule, `RULE { template } WHERE { pattern }`, is like SPARQL's `CONSTRUCT { template } WHERE { pattern }`. The `WHERE` clause matches the input graph, the template creates triples, and the new triples are returned as a separate graph. The input graph is not changed. A rule is also similar to SPARQL's `INSERT { template } WHERE { pattern }`, which creates the same triples but writes them to the graph it reads. None of the three can delete triples, unlike SPARQL's `DELETE`.

The head template of an SRL rule and the template of a `CONSTRUCT` are both templates of triples. They can use variables, blank nodes and RDF 1.2 syntax (annotations, reifiers and triple terms). A blank node is a new node for each match, and a triple that is not valid, such as one with a literal as its subject, is dropped.

The cell below runs the same template in a `CONSTRUCT` and in an SRL rule, over the graph from section 04.04.1. Each creates an annotated `ex:exposedTo` triple for every `ex:hasVulnerability` triple in the input graph, and the cell compares the triples each one creates:

In [4]:
construct_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:exposedTo ?v {| ex:derivedBy ex:R1 |} }
WHERE     { ?x ex:hasVulnerability ?v }
"""

rule = """
PREFIX ex: <http://example.org/>
RULE  { ?x ex:exposedTo ?v {| ex:derivedBy ex:R1 |} }
WHERE { ?x ex:hasVulnerability ?v }
"""

created_by_construct = base.query(construct_query).graph
created_by_rule = srl.parse_ruleset(rule).infer(base)

print("triples created by the CONSTRUCT:", len(created_by_construct))
print("triples created by the rule:     ", len(created_by_rule))
print("same triples:", isomorphic(created_by_construct, created_by_rule))

triples created by the CONSTRUCT: 3
triples created by the rule:      3
same triples: True


## 04.04.3 SRL, Datasets and Remote Endpoints

A SPARQL query can work with a dataset, which is a default graph and named graphs. `GRAPH`, `FROM` and `FROM NAMED` choose the graphs a pattern is matched against. The SRL specification works with one input graph and one inference graph. Accordingly, it has no `GRAPH`, `FROM` or `FROM NAMED` expression, and a rule cannot write to a named graph.

StarLayer accepts a dataset as input to SRL inference and runs the rules on the union of the dataset's graphs, so a rule can match triples from different graphs.

**Remote endpoints.** SPARQL's `SERVICE` lets a query read from a remote endpoint as part of its pattern. SRL does not support `SERVICE`, so a rule cannot use a remote endpoint as the input graph. The typical approach is to query the endpoint before running the rules. Use a `CONSTRUCT` against the endpoint to get the triples the rules need, put them in a graph, and then use that graph as the input graph.

## 04.04.4 How a rule runs differently

- **New Triples.** An SRL rule only creates new triples in the inference graph. Triples already present in the base graph are not created. A `CONSTRUCT` creates new triples without consideration of what is in the input graph.
- **Repetition.** A rule is applied again until no new triples are created, so it sees the triples that rules, itself included, have added. A `CONSTRUCT` runs once each time it is called.
- **Order.** `CONSTRUCT` queries run in the order they are called. Rules run in the order of a rule plan that the engine builds from them. The order the rules are written does not matter.


## 04.04.5 Names and `DATA`

**Names** 

An SRL rule can be named with an IRI after `RULE`. `RULE ex:R1 { ... } WHERE { ... }` names the rule `ex:R1`. The name identifies the rule in the plan and its dependency graph, and can be written in the head to record which rule derived a triple ([04.03](04.03-srl-more-features.ipynb), section 04.03.2). A `CONSTRUCT` has no name.

**DATA**

An SRL rule can contain a `DATA { }` block. It holds triples with no variables, and they are added to the inference graph if they are not already in the base graph. A `CONSTRUCT` has no `DATA` block, but can write triples directly to the output graph.  

The cell below writes two triples as a `DATA` block, and as a `CONSTRUCT` with an empty `WHERE`. One of them is already in the base graph. The SRL rule only writes new triples to the inference graph. A `CONSTRUCT` writes all triples so it includes the triple in the base graph.

In [5]:
data_rules = """
PREFIX ex: <http://example.org/>
DATA {
    ex:logger ex:hasVulnerability ex:vuln2 .
    ex:app ex:dependsOn ex:db .
}
"""

construct_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT {
    ex:logger ex:hasVulnerability ex:vuln2 .
    ex:app ex:dependsOn ex:db .
} WHERE {}
"""

from_data = srl.parse_ruleset(data_rules).infer(base)
from_data.bind("ex", EX)
from_construct = base.query(construct_query).graph
from_construct.bind("ex", EX)

print("inference graph of the DATA block")
show(from_data)
print("result of the CONSTRUCT")
show(from_construct)

inference graph of the DATA block
@prefix ex: <http://example.org/> .

ex:logger ex:hasVulnerability ex:vuln2 .

result of the CONSTRUCT
@prefix ex: <http://example.org/> .

ex:app ex:dependsOn ex:db .

ex:logger ex:hasVulnerability ex:vuln2 .



## 04.04.6 `SET` and `BIND`

SPARQL has `BIND(expression AS ?v)`, which assigns the value of an expression to a new variable. SRL uses `SET(?v := expression)`.

They work alike. Each computes one value for every solution, from the variables bound before it, and the variable must be a new one that the pattern has not used. Both add a new column to the existing solution.

They differ when the expression is an error, such as adding a number to a string. `BIND` keeps the solution and leaves the variable unbound. Because SRL does not permit unbound variables in a solution, `SET` drops the solution when the expression is an error. The SRL specification says `SET(?v := expr)` is the same as `BIND(expr AS ?v)` followed by `FILTER(BOUND(?v))`. 

In this example one person's age is a string. The same pattern is written as a `CONSTRUCT` with `BIND` and as a rule with `SET`, and each head has the same two triples. 

For `ex:bob`, `?age + 1` is an error, because it adds a number to a string ("unknown"). With `BIND`, the solution row is kept and `?next` is unbound. The `CONSTRUCT` then creates the triple that does not use `?next`, `ex:bob ex:checked true`, and skips the one that does, because a triple with an unbound variable is not created. With `SET`, the whole solution row is dropped, so the rule makes neither triple for `ex:bob`.

In [6]:
people = StarLayerGraph(prefixes={"ex": EX})
people.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:ann ex:age 30 .
    ex:bob ex:age "unknown" .
""", format="turtle")

head = "{ ?p ex:checked true . ?p ex:nextAge ?next }"

created_by_construct = people.query(
    f"PREFIX ex: <{EX}> CONSTRUCT {head} WHERE {{ ?p ex:age ?age . BIND(?age + 1 AS ?next) }}"
).graph
created_by_construct.bind("ex", EX)

created_by_rule = srl.parse_ruleset(f"PREFIX ex: <{EX}> RULE {head} WHERE {{ ?p ex:age ?age SET(?next := ?age + 1) }}").infer(people)
created_by_rule.bind("ex", EX)

print("created by the CONSTRUCT with BIND")
show(created_by_construct)
print("created by the rule with SET")
show(created_by_rule)

created by the CONSTRUCT with BIND
@prefix ex: <http://example.org/> .

ex:ann ex:checked true ;
    ex:nextAge 31 .

ex:bob ex:checked true .

created by the rule with SET
@prefix ex: <http://example.org/> .

ex:ann ex:checked true ;
    ex:nextAge 31 .



## 04.04.7 Property paths

SPARQL builds a path from predicates with seven operators and parentheses:

- `/` is a sequence. `ex:a/ex:b` follows `ex:a`, then `ex:b`.
- `|` is an alternative. `ex:a|ex:b` follows either predicate.
- `^` is the inverse. `^ex:a` follows `ex:a` backwards, so `?x ^ex:a ?y` matches the same triples as `?y ex:a ?x`.
- `*` is zero or more steps. `ex:a*` reaches the start node itself, then everything reached by one step, two steps and so on.
- `+` is one or more steps. `ex:a+` is `ex:a*` without the start node itself.
- `?` is zero or one step. `ex:a?` reaches the start node itself and whatever one step reaches.
- `!` is a negation. `!ex:a` follows one step along any predicate except `ex:a`, and `!(ex:a|ex:b)` excludes both. It matches a single step, never a longer path.
- `( )` groups part of a path, so that an operator applies to the whole group. `^(ex:a/ex:b)` follows `ex:a` then `ex:b`, backwards, where `^ex:a/ex:b` reverses only the first step.

SRL allows only `/` and `^`, and parentheses.

SRL accepts these operators in a rule body, so the same path works in a `CONSTRUCT` and in a rule. This path goes from a vulnerability back to each component that depends on a component that has it. It uses all three: `/` for the sequence, `^` for the inverse, and parentheses to group the sequence before it is reversed:

In [6]:
# SPARQL: the path in a CONSTRUCT
by_query = base.query("""
    PREFIX ex: <http://example.org/>
    CONSTRUCT { ?v ex:affects ?x } WHERE { ?v ^(ex:dependsOn/ex:hasVulnerability) ?x }
""").graph

# SRL: the same path in a rule
by_rule = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?v ex:affects ?x } WHERE { ?v ^(ex:dependsOn/ex:hasVulnerability) ?x }
""").infer(base)

by_rule.bind("ex", EX)
show(by_rule)
print("same triples as the query:", isomorphic(by_query, by_rule))

@prefix ex: <http://example.org/> .

ex:vuln1 ex:affects ex:app .

same triples as the query: True


The other operators are not allowed in SRL. The SRL specification says that arbitrary-length paths (`*` and `+`) can be expressed with rule recursion, which is a more general approach. An alternative, `a|b`, can be written as two rules. Likewise `?` (zero or one step) and `!` (negation) can be expressed as a series of rules.

The following code sections demonstrate rule-based versions of the unsupported property path operators. Each builds a small graph, runs a SPARQL `CONSTRUCT` query that uses the operator, then runs rules that produce the same triples and compares the two.

**`+`: one or more steps.** The graph is a chain from `a` to `d`. The query finds everything each node reaches. The rules say it with recursion: the first covers one step, and the second adds a step to anything already reached:

In [7]:
chain = StarLayerGraph(prefixes={"ex": EX})
chain.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:a ex:next ex:b .
    ex:b ex:next ex:c .
    ex:c ex:next ex:d .
""", format="turtle")

# SPARQL: the operator does the repeating
by_operator = chain.query("""
    PREFIX ex: <http://example.org/>
    CONSTRUCT { ?x ex:reaches ?y } WHERE { ?x ex:next+ ?y }
""").graph

# SRL: the second rule reads the triples the rules have already derived
by_rules = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?x ex:reaches ?y } WHERE { ?x ex:next ?y }
    RULE { ?x ex:reaches ?z } WHERE { ?x ex:reaches ?y . ?y ex:next ?z }
""").infer(chain)

by_operator.bind("ex", EX)
show(by_operator)
print("same triples as the query:", isomorphic(by_operator, by_rules))

@prefix ex: <http://example.org/> .

ex:a ex:reaches ex:b, ex:c, ex:d .

ex:b ex:reaches ex:c, ex:d .

ex:c ex:reaches ex:d .

same triples as the query: True


**`|`: one or the other.** The graph records mothers and fathers. The query finds the parent of each person by either predicate. The rules use one rule for each alternative, with the same head:

In [8]:
family = StarLayerGraph(prefixes={"ex": EX})
family.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:ann ex:mother ex:bea .
    ex:ann ex:father ex:carl .
    ex:bea ex:mother ex:dee .
""", format="turtle")

# SPARQL: the operator chooses between the predicates
by_operator = family.query("""
    PREFIX ex: <http://example.org/>
    CONSTRUCT { ?x ex:parent ?y } WHERE { ?x ex:mother|ex:father ?y }
""").graph

# SRL: one rule for each alternative, with the same head
by_rules = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?x ex:parent ?y } WHERE { ?x ex:mother ?y }
    RULE { ?x ex:parent ?y } WHERE { ?x ex:father ?y }
""").infer(family)

by_operator.bind("ex", EX)
show(by_operator)
print("same triples as the query:", isomorphic(by_operator, by_rules))

@prefix ex: <http://example.org/> .

ex:ann ex:parent ex:bea, ex:carl .

ex:bea ex:parent ex:dee .

same triples as the query: True


**`?`: zero or one step.** `ex:next?` follows `ex:next` zero times or one time. Zero times means a node reaches itself, and that holds for every node in the graph, even one with no `ex:next` triple. One time reaches the next node. The graph below has a chain from `a` to `c` and a node `z` that has no `ex:next`.

To replicate the `CONSTRUCT`, SRL needs three rules. Two make every node reach itself, one for nodes that are a subject and one for nodes that are an object. The third covers the one step. `WHERE GROUND` matches the base graph only, so the first two rules do not read the triples the rules themselves derive. That matters when a rule's head creates a new node, such as `?x ex:reaches ex:newNode`. A plain `WHERE` would read that derived triple, and `ex:newNode` would reach itself too. In the cell, each head also links the node it reaches to `ex:newNode`, a node that is not in the base graph:

In [8]:
steps = StarLayerGraph(prefixes={"ex": EX})
steps.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:a ex:next ex:b .
    ex:b ex:next ex:c .
    ex:z ex:name "Z" .
""", format="turtle")

# SPARQL: the operator allows zero steps or one step
by_operator = steps.query("""
    PREFIX ex: <http://example.org/>
    CONSTRUCT { ?x ex:reaches ?y . ?y ex:via ex:newNode } WHERE { ?x ex:next? ?y }
""").graph

# SRL: two rules for the zero steps, one rule for the one step
by_rules = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?s ex:reaches ?s . ?s ex:via ex:newNode } WHERE GROUND { ?s ?p ?o }
    RULE { ?o ex:reaches ?o . ?o ex:via ex:newNode } WHERE GROUND { ?s ?p ?o }
    RULE { ?x ex:reaches ?y . ?y ex:via ex:newNode } WHERE { ?x ex:next ?y }
""").infer(steps)

by_operator.bind("ex", EX)
show(by_operator)
print("same triples as the query:", isomorphic(by_operator, by_rules))

@prefix ex: <http://example.org/> .

ex:a ex:reaches ex:a, ex:b ;
    ex:via ex:newNode .

ex:b ex:reaches ex:b, ex:c ;
    ex:via ex:newNode .

ex:c ex:reaches ex:c ;
    ex:via ex:newNode .

ex:z ex:reaches ex:z ;
    ex:via ex:newNode .

same triples as the query: True


**`*`: zero or more steps.** `ex:next*` is `ex:next?` that keeps going. It uses the same two zero-step rules, with `WHERE GROUND` for the same reason. The recursive rule replaces the one-step rule, because the zero-step rules already give each node a `reaches` triple to extend. It uses the graph from the `?` cell:

In [9]:
# the graph from the cell above: a chain from a to c, and a node z with no ex:next

# SPARQL: the operator allows zero or more steps
by_operator = steps.query("""
    PREFIX ex: <http://example.org/>
    CONSTRUCT { ?x ex:reaches ?y } WHERE { ?x ex:next* ?y }
""").graph

# SRL: two rules for the zero steps, and a rule that adds one step to anything already reached
by_rules = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?s ex:reaches ?s } WHERE GROUND { ?s ?p ?o }
    RULE { ?o ex:reaches ?o } WHERE GROUND { ?s ?p ?o }
    RULE { ?x ex:reaches ?z } WHERE { ?x ex:reaches ?y . ?y ex:next ?z }
""").infer(steps)

by_operator.bind("ex", EX)
show(by_operator)
print("same triples as the query:", isomorphic(by_operator, by_rules))

@prefix ex: <http://example.org/> .

ex:a ex:reaches ex:a, ex:b, ex:c .

ex:b ex:reaches ex:b, ex:c .

ex:c ex:reaches ex:c .

ex:z ex:reaches ex:z .

same triples as the query: True


**`!`: any predicate except these.** The query follows every predicate from `ex:ann` except `ex:name`. The rule matches any predicate with a variable and excludes `ex:name` with a `FILTER`. To exclude several predicates, use `FILTER(?p NOT IN (ex:mother, ex:name))`:

In [10]:
ann = StarLayerGraph(prefixes={"ex": EX})
ann.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:ann ex:mother ex:bea .
    ex:ann ex:father ex:carl .
    ex:ann ex:name "Ann" .
""", format="turtle")

# SPARQL: the operator excludes a predicate
by_operator = ann.query("""
    PREFIX ex: <http://example.org/>
    CONSTRUCT { ex:ann ex:relative ?y } WHERE { ex:ann !ex:name ?y }
""").graph

# SRL: a variable for the predicate, and a FILTER that excludes it
by_rule = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ex:ann ex:relative ?y } WHERE { ex:ann ?p ?y FILTER(?p != ex:name) }
""").infer(ann)

by_operator.bind("ex", EX)
show(by_operator)
print("same triples as the query:", isomorphic(by_operator, by_rule))

@prefix ex: <http://example.org/> .

ex:ann ex:relative ex:bea, ex:carl .

same triples as the query: True


## 04.04.8 `WHERE GROUND` and `NOT GROUND`

A plain `WHERE` in an SRL rule sees every triple the rules can read: the base graph, the rule set's own `DATA` triples and the derived triples. `WHERE GROUND` matches the base graph only. This allows an SRL rule to only act on data in the base graph and ignore data from the rule set's own `DATA` triples and derived triples. Also, inside a rule WHERE clause, `NOT GROUND { pattern }` checks the base graph only for one negation. 

A `CONSTRUCT` reads only the input graph so it does not need expressions to ignore derived triples.  



## 04.04.9 `EXISTS`

In SPARQL, `EXISTS { pattern }` is a function that is true when the pattern has a match for the current solution. It is typically used in one of two ways:
- Filter a solution based on a pattern - `FILTER EXISTS { pattern }`, and 
- Assign a boolean variable based on a pattern - `BIND(EXISTS { pattern } AS ?has)` 

SRL does not include `EXISTS`. A rule obtains the same behavior in two ways:

- **Filter a solution.** Write the matching pattern to bind a variable in the solution. The solution is filtered to those rows matching the pattern. This approach works with SPARQL as well. If the bound variable is not needed it is ignored in the rule head. In SPARQL `FILTER EXISTS` allows the filter to operate without binding a variable.  
- **Assign a boolean.** In SRL two rules can be written with the same head. One has the pattern in its body and gives `true`. The other has `NOT { pattern }` and gives `false`.

The following code uses components where `ex:app` has two owners, `ex:db` has one and `ex:logger` has none. It compares a `CONSTRUCT` with a rule for the first two uses. Both produce the same set of triples.

In [11]:
owners = StarLayerGraph(prefixes={"ex": EX})
owners.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:app    a ex:Component ; ex:owner ex:teamA, ex:teamC .
    ex:db     a ex:Component ; ex:owner ex:teamB .
    ex:logger a ex:Component .
""", format="turtle")

# As a condition: the components that have an owner
condition_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:owned true }
WHERE { ?x a ex:Component FILTER EXISTS { ?x ex:owner ?o } }
"""

condition_rule = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:owned true }
WHERE { ?x a ex:Component . ?x ex:owner ?o }
"""

by_construct = owners.query(condition_query).graph
by_construct.bind("ex", EX)
by_rule = srl.parse_ruleset(condition_rule).infer(owners)
by_rule.bind("ex", EX)

print("as a condition: triples created by the CONSTRUCT")
show(by_construct)
print("as a condition: triples created by the rule")
show(by_rule)

# As a value: every component, with true or false
value_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:hasOwner ?has }
WHERE { ?x a ex:Component BIND(EXISTS { ?x ex:owner ?o } AS ?has) }
"""

value_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:hasOwner true }  WHERE { ?x a ex:Component . ?x ex:owner ?o }
RULE { ?x ex:hasOwner false } WHERE { ?x a ex:Component NOT { ?x ex:owner ?o } }
"""

by_construct = owners.query(value_query).graph
by_construct.bind("ex", EX)
by_rules = srl.parse_ruleset(value_rules).infer(owners)
by_rules.bind("ex", EX)

print("as a value: triples created by the CONSTRUCT")
show(by_construct)
print("as a value: triples created by the rules")
show(by_rules)

as a condition: triples created by the CONSTRUCT
@prefix ex: <http://example.org/> .

ex:app ex:owned true .

ex:db ex:owned true .

as a condition: triples created by the rule
@prefix ex: <http://example.org/> .

ex:app ex:owned true .

ex:db ex:owned true .

as a value: triples created by the CONSTRUCT
@prefix ex: <http://example.org/> .

ex:app ex:hasOwner true .

ex:db ex:hasOwner true .

ex:logger ex:hasOwner false .

as a value: triples created by the rules
@prefix ex: <http://example.org/> .

ex:app ex:hasOwner true .

ex:db ex:hasOwner true .

ex:logger ex:hasOwner false .



## 04.04.10 `NOT` and `FILTER NOT EXISTS`

SRL's `NOT { pattern }` and SPARQL's `FILTER NOT EXISTS { pattern }` both retain a solution only if the pattern has no match. They differ in two ways:

- **Position.** `NOT` is checked at the point where it is written, so it can only use the variables bound before it and does not impact subsequent statements. A SPARQL `FILTER` is checked after the whole group is matched, so it can be written anywhere in the query.
- **What goes inside.** `NOT` takes triple patterns and filters only, with no nested patterns and no additional negations. `FILTER NOT EXISTS` can hold any pattern, such as an `OPTIONAL` nested pattern.


In [12]:
owners = StarLayerGraph(prefixes={"ex": EX})
owners.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:app    a ex:Component ; ex:owner ex:teamA .
    ex:db     a ex:Component ; ex:owner ex:teamB .
    ex:logger a ex:Component .
""", format="turtle")

# SPARQL: the components not owned by ex:teamA. The FILTER can come first, because it is checked after ?x is bound
construct_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:flag true }
WHERE {
    FILTER NOT EXISTS { ?x ex:owner ?o FILTER(?o = ex:teamA) }
    ?x a ex:Component
}
"""

# SRL: the same, with a FILTER inside the NOT. The NOT must come second, after the pattern that binds ?x
rule = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:flag true }
WHERE {
    ?x a ex:Component
    NOT { ?x ex:owner ?o FILTER(?o = ex:teamA) }
}
"""

by_construct = owners.query(construct_query).graph
by_rule = srl.parse_ruleset(rule).infer(owners)

print("triples created by the CONSTRUCT:", len(by_construct))
print("triples created by the rule:     ", len(by_rule))

triples created by the CONSTRUCT: 2
triples created by the rule:      2


`NOT` cannot hold a `UNION` or a nested `NOT EXISTS`. Two techniques cover them:

- **A `UNION` becomes one `NOT` for each alternative.** "Neither this nor that" can be expressed as "not this, and not that". `FILTER NOT EXISTS { { ?x ex:owner ?o } UNION { ?x ex:steward ?o } }` is `NOT { ?x ex:owner ?owner } NOT { ?x ex:steward ?steward }`. Each `NOT` needs its own variable names: since the draft of November 2026, a name used inside one `NOT` may not appear in another part of the rule.
- **A nested negation takes a helper rule.** For the components whose owners are all teams, the first rule derives `?x ex:hasNonTeamOwner true` from `?x ex:owner ?o NOT { ?o a ex:Team }`. The second rule is `NOT { ?x ex:hasNonTeamOwner true }`. The helper rule runs first, because the second depends on it. A rule cannot delete triples, so the helper triple stays in the inference graph. If the rule annotates it, as in `{| ex:helper true |}`, a SPARQL update can delete it from the result afterwards.

A join is several triple patterns that share a variable. A join goes inside a `NOT` as it is. In `NOT { ?x ex:owner ?o . ?o a ex:Team }`, the variable `?o` links the two patterns, so it finds the components that have a team as an owner.

The code shows all three, each as a `CONSTRUCT` and as rules, and compares the triples. The nested rules annotate the helper triple, and the cell deletes it with an update before it compares the triples.

In [ ]:
managed = StarLayerGraph(prefixes={"ex": EX})
managed.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:app    a ex:Component ; ex:owner ex:teamA .
    ex:db     a ex:Component ; ex:owner ex:bob .
    ex:cache  a ex:Component ; ex:steward ex:carol .
    ex:logger a ex:Component .
    ex:teamA  a ex:Team .
""", format="turtle")

# A UNION inside NOT EXISTS: the components with no owner and no steward
union_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:unmanaged true }
WHERE {
    ?x a ex:Component
    FILTER NOT EXISTS { { ?x ex:owner ?o } UNION { ?x ex:steward ?o } }
}
"""

# SRL: one NOT for each alternative
union_rule = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:unmanaged true }
WHERE {
    ?x a ex:Component
    NOT { ?x ex:owner ?owner }
    NOT { ?x ex:steward ?steward }
}
"""

# A nested NOT EXISTS: the components whose owners are all teams
nested_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:onlyTeamOwners true }
WHERE {
    ?x a ex:Component
    FILTER NOT EXISTS { ?x ex:owner ?o FILTER NOT EXISTS { ?o a ex:Team } }
}
"""

# SRL: a helper rule finds the owners that are not teams, and the second rule negates it
# the helper triple is annotated so it can be easily delted.
nested_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:hasNonTeamOwner true {| ex:helper true |} } WHERE { ?x ex:owner ?o NOT { ?o a ex:Team } }
RULE { ?x ex:onlyTeamOwners true }  WHERE { ?x a ex:Component NOT { ?x ex:hasNonTeamOwner true } }
"""

# A join inside NOT EXISTS: the components that do not have a team as an owner
join_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:noTeamOwner true }
WHERE {
    ?x a ex:Component
    FILTER NOT EXISTS { ?x ex:owner ?o . ?o a ex:Team }
}
"""

# SRL: the join goes inside the NOT as it is
join_rule = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:noTeamOwner true }
WHERE {
    ?x a ex:Component
    NOT { ?x ex:owner ?o . ?o a ex:Team }
}
"""

union_by_construct = managed.query(union_query).graph
union_by_rule = srl.parse_ruleset(union_rule).infer(managed)
print("union, same triples:", isomorphic(union_by_construct, union_by_rule))

nested_by_construct = managed.query(nested_query).graph
nested_by_rules = srl.parse_ruleset(nested_rules).infer(managed)
print("nested, triples in the inference graph:", len(nested_by_rules))

# the helper triple is annotated ex:helper, so an update can delete it and its annotation
nested_by_rules.update("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
DELETE { ?s ?p ?o . ?r rdf:reifies <<( ?s ?p ?o )>> . ?r ex:helper true }
WHERE { ?r ex:helper true . ?r rdf:reifies <<( ?s ?p ?o )>> . ?s ?p ?o }
""")
print("nested, triples after deleting the helper triples:", len(nested_by_rules))
print("nested, same triples:", isomorphic(nested_by_construct, nested_by_rules))

join_by_construct = managed.query(join_query).graph
join_by_rule = srl.parse_ruleset(join_rule).infer(managed)
print("join, same triples:", isomorphic(join_by_construct, join_by_rule))

union, same triples: True
nested, triples in the inference graph: 6
nested, triples after deleting the helper triples: 3
nested, same triples: True
join, same triples: True


## 04.04.11 Other expressions


### 04.04.11.1 `OPTIONAL`, `COALESCE` and `BOUND`

The SPARQL `OPTIONAL { pattern }` expression adds the variables of the pattern when the pattern matches, and leaves them unbound when it does not. SPARQL then uses `COALESCE` to assign a value to an unbound variable, and `BOUND(?v)` to test whether a variable has a value. An SRL rule cannot leave a variable unbound, so it has no `OPTIONAL`, `COALESCE` or `BOUND` expression. To obtain equivalent results in SRL rules, the work can be split among rules based on what each triple needs:

- **Triples that do not use the optional variables** go in a rule that does not mention the optional pattern.
- **Triples that use the optional variables** go in a rule that matches the optional pattern as an ordinary pattern.
- **A default value** for the case with no match goes in a second rule with `NOT { pattern }` and the default is set as a constant in the head.
- **A `BOUND` test** becomes the same two rules. The rule that matches the pattern gives `true`, and the rule with `NOT { pattern }` gives `false`.

The following code uses components where `ex:app` has an owner and the others do not. It compares a `CONSTRUCT` that uses `OPTIONAL` and `COALESCE` to give each component a contact, with the default `ex:unassigned`, and uses `BOUND` to record whether it has an owner, against rules that do the same:

In [15]:
listed = StarLayerGraph(prefixes={"ex": EX})
listed.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:app    a ex:Component ; ex:owner ex:teamA .
    ex:db     a ex:Component .
    ex:logger a ex:Component .
""", format="turtle")

# SPARQL: a default contact with COALESCE, and a flag with BOUND
default_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:contact ?contact . ?x ex:hasOwner ?has }
WHERE {
    ?x a ex:Component
    OPTIONAL { ?x ex:owner ?o }
    BIND(COALESCE(?o, ex:unassigned) AS ?contact)
    BIND(BOUND(?o) AS ?has)
}
"""

# SRL: a rule for the owner, and a rule with NOT for the default
default_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:contact ?o . ?x ex:hasOwner true } WHERE { ?x a ex:Component . ?x ex:owner ?o }
RULE { ?x ex:contact ex:unassigned . ?x ex:hasOwner false } WHERE { ?x a ex:Component NOT { ?x ex:owner ?o } }
"""

by_construct = listed.query(default_query).graph
by_rules = srl.parse_ruleset(default_rules).infer(listed)
by_rules.bind("ex", EX)
show(by_rules)
print("default contact, same triples:", isomorphic(by_construct, by_rules))

@prefix ex: <http://example.org/> .

ex:app ex:contact ex:teamA ;
    ex:hasOwner true .

ex:db ex:contact ex:unassigned ;
    ex:hasOwner false .

ex:logger ex:contact ex:unassigned ;
    ex:hasOwner false .

default contact, same triples: True


### 04.04.11.2 `UNION`

In SPARQL `{ pattern A } UNION { pattern B }` gives the solutions of A and the solutions of B. The two branches can bind different variables, so a solution from one branch can leave the other branch's variables unbound. An SRL rule cannot leave a variable unbound, so it has no `UNION` expression. In SRL, one rule can be used for each branch with the same head. Each rule matches only its own branch, so each binds every variable its head uses.

The following code uses components that have an owner, a steward, both, or neither. It compares a `CONSTRUCT` with a `UNION` against rules, one for each branch:

In [ ]:
staffed = StarLayerGraph(prefixes={"ex": EX})
staffed.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:app    a ex:Component ; ex:owner ex:teamA .
    ex:db     a ex:Component ; ex:steward ex:bob .
    ex:cache  a ex:Component ; ex:owner ex:teamB ; ex:steward ex:carol .
    ex:logger a ex:Component .
""", format="turtle")

# SPARQL: the contact is the owner or the steward
union_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:contact ?c }
WHERE {
    ?x a ex:Component .
    { ?x ex:owner ?c } 
    UNION 
    { ?x ex:steward ?c }
}
"""

# SRL: one rule for each branch, with the same head
union_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:contact ?c } WHERE { ?x a ex:Component . ?x ex:owner ?c }
RULE { ?x ex:contact ?c } WHERE { ?x a ex:Component . ?x ex:steward ?c }
"""

by_construct = staffed.query(union_query).graph
by_rules = srl.parse_ruleset(union_rules).infer(staffed)
by_rules.bind("ex", EX)
show(by_rules)
print("union, same triples:", isomorphic(by_construct, by_rules))

@prefix ex: <http://example.org/> .

ex:app ex:contact ex:teamA .

ex:cache ex:contact ex:carol, ex:teamB .

ex:db ex:contact ex:bob .

union, same triples: True


### 04.04.11.3 `MINUS`

The SPARQL `MINUS { pattern }` removes the solutions that the pattern matches. A rule uses `NOT { pattern }` for this. They agree when the pattern shares a variable with the rest of the body. `MINUS { ?x ex:owner ?o }` and `NOT { ?x ex:owner ?o }` both remove the components that have an owner.

They differ when the pattern shares no variable with the rest of the body. SPARQL `MINUS` with no shared variables removes nothing, while SRL `NOT` removes every solution, assuming the pattern matches somewhere in the graph. 

The following code uses components where `ex:app` has an owner, `ex:db` has a steward and `ex:logger` has neither, and `ex:teamA` is a team. It compares `MINUS` with `NOT`, first for a pattern that shares `?x`, then for a pattern that shares no variable:

In [17]:
catalog = StarLayerGraph(prefixes={"ex": EX})
catalog.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:app    a ex:Component ; ex:owner ex:teamA .
    ex:db     a ex:Component ; ex:steward ex:bob .
    ex:logger a ex:Component .
    ex:teamA  a ex:Team .
""", format="turtle")

# A pattern that shares ?x: the components with no owner
shared_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:unowned true }
WHERE { ?x a ex:Component MINUS { ?x ex:owner ?o } }
"""

shared_rule = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:unowned true } WHERE { ?x a ex:Component NOT { ?x ex:owner ?o } }
"""

by_construct = catalog.query(shared_query).graph
by_rule = srl.parse_ruleset(shared_rule).infer(catalog)
print("shared variable, same triples:", isomorphic(by_construct, by_rule))

# A pattern that shares no variable with the rest of the body
separate_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:listed true }
WHERE { ?x a ex:Component MINUS { ?t a ex:Team } }
"""

separate_rule = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:listed true } WHERE { ?x a ex:Component NOT { ?t a ex:Team } }
"""

by_construct = catalog.query(separate_query).graph
by_rule = srl.parse_ruleset(separate_rule).infer(catalog)
print("no shared variable, triples from MINUS:", len(by_construct))
print("no shared variable, triples from NOT:  ", len(by_rule))

shared variable, same triples: True
no shared variable, triples from MINUS: 3
no shared variable, triples from NOT:   0


### 04.04.11.4 `VALUES`

SPARQL `VALUES ?v { a b }` supplies a fixed list of values and joins them with the rest of the pattern, so each solution is repeated once for each value. SRL does not support `VALUES`. Alternatively, the values can be placed in the rule set as triples in a `DATA` block allowing them to be matched in the body of rules. 

Translating SPARQL `VALUES` to an SRL `DATA` block, requires one triple for each value, preferably with an annotation that allows the values to be deleted from the inference graph.   

The following code uses two boxes with a width. The heights 10 and 20 are values, and the area of each box is its width times each height. The `CONSTRUCT` lists the heights with `VALUES`, and the rule set holds them as `DATA`. The `DATA` triples are annotated `ex:temporary`, and the cell deletes them from the inference graph with an update before it compares the triples:

In [18]:
boxes = StarLayerGraph(prefixes={"ex": EX})
boxes.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:box1 ex:width 2 .
    ex:box2 ex:width 3 .
""", format="turtle")

# SPARQL: VALUES gives the heights
values_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?b ex:area ?area }
WHERE {
    ?b ex:width ?w
    VALUES ?h { 10 20 }
    BIND(?w * ?h AS ?area)
}
"""

# SRL: the heights are annotated DATA triples, and the body matches them
values_rules = """
PREFIX ex: <http://example.org/>
DATA {
    ex:options ex:height 10 {| ex:temporary true |} .
    ex:options ex:height 20 {| ex:temporary true |} .
}
RULE { ?b ex:area ?area }
WHERE {
    ?b ex:width ?w .
    ex:options ex:height ?h
    SET(?area := ?w * ?h)
}
"""

by_construct = boxes.query(values_query).graph
by_rules = srl.parse_ruleset(values_rules).infer(boxes)
print("triples in the inference graph:", len(by_rules))

# delete the annotated DATA triples and their annotations
by_rules.update("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
DELETE { ?s ?p ?o . ?r rdf:reifies <<( ?s ?p ?o )>> . ?r ex:temporary true }
WHERE { ?r ex:temporary true . ?r rdf:reifies <<( ?s ?p ?o )>> . ?s ?p ?o }
""")
print("triples after deleting them:", len(by_rules))

by_rules.bind("ex", EX)
show(by_rules)
print("same triples:", isomorphic(by_construct, by_rules))

triples in the inference graph: 10
triples after deleting them: 4
@prefix ex: <http://example.org/> .

ex:box1 ex:area 20, 40 .

ex:box2 ex:area 30, 60 .

same triples: True


### 04.04.11.5 Sub-selects

A SPARQL sub-select is a `SELECT` query inside a `WHERE`. It computes a set of solutions that the outer pattern joins with. SRL has no sub-selects. 

An SRL rule set can replicate the behavior of a sub-select with multiple rules that create interim triples, which are then used to generate the final result. The interim triples can be deleted after the rule set is complete if they are not needed.

The following code finds the components with the most vulnerabilities. The `CONSTRUCT` uses a sub-select that counts the vulnerabilities of each component, and another that takes the maximum of those counts. The rules derive the counts as annotated triples, and a second rule takes the maximum of them with a group element. The cell deletes the annotated triples before it compares the results:

In [18]:
exposure = StarLayerGraph(prefixes={"ex": EX})
exposure.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:app    a ex:Component ; ex:hasVulnerability ex:v1, ex:v2, ex:v3 .
    ex:cache  a ex:Component ; ex:hasVulnerability ex:v4, ex:v5 .
    ex:db     a ex:Component ; ex:hasVulnerability ex:v6 .
    ex:logger a ex:Component .
""", format="turtle")

# SPARQL: one sub-select counts for each component, another takes the maximum of the counts
subselect_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:mostExposed true }
WHERE {
    { SELECT ?x (COUNT(?v) AS ?n) WHERE { ?x ex:hasVulnerability ?v } GROUP BY ?x }
    { SELECT (MAX(?m) AS ?max)
      WHERE { SELECT ?y (COUNT(?w) AS ?m) WHERE { ?y ex:hasVulnerability ?w } GROUP BY ?y } }
    FILTER(?n = ?max)
}
"""

# SRL: the counts are annotated triples, and the second rule takes the maximum of them
subselect_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:vulnerabilityCount ?n {| ex:temporary true |} }
WHERE {
    ?x a ex:Component .
    GROUP BY ?x { ?x ex:hasVulnerability ?v } [ ?n := COUNT(*) ]
}

RULE { ?x ex:mostExposed true }
WHERE {
    ?x ex:vulnerabilityCount ?n .
    GROUP { ?y ex:vulnerabilityCount ?c } [ ?max := MAX(?c) ]
    FILTER(?n = ?max)
}
"""

by_construct = exposure.query(subselect_query).graph
by_rules = srl.parse_ruleset(subselect_rules).infer(exposure)
print("triples in the inference graph:", len(by_rules))

# delete the annotated count triples and their annotations
by_rules.update("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
DELETE { ?s ?p ?o . ?r rdf:reifies <<( ?s ?p ?o )>> . ?r ex:temporary true }
WHERE { ?r ex:temporary true . ?r rdf:reifies <<( ?s ?p ?o )>> . ?s ?p ?o }
""")
print("triples after deleting them:", len(by_rules))

by_rules.bind("ex", EX)
show(by_rules)
print("same triples:", isomorphic(by_construct, by_rules))

triples in the inference graph: 10
triples after deleting them: 1
@prefix ex: <http://example.org/> .

ex:app ex:mostExposed true .

same triples: True


### 04.04.11.6 Aggregates in expressions

In SPARQL an aggregate can be part of an expression, such as `SUM(?p * ?q) + 5` or `HAVING (COUNT(?v) > 1)`. In SRL an aggregate is allowed only in a group element, `[ ?sum := SUM(?amount) ]`, and its argument must be a variable. An aggregate inside a `FILTER` or `SET` expression is rejected when the rule set is parsed, with `aggregate_in_expression`. The two cases take these steps:

- **An expression inside the aggregate**, such as `SUM(?p * ?q)`. Compute it with `SET` in the group pattern, and aggregate the variable.
- **An expression around the aggregate**, such as `SUM(...) + 5`. Store the aggregate in a variable, then compute with `SET` after the group element.

The following code totals each order from the price and quantity of its lines and adds 5 for shipping. The `CONSTRUCT` writes the whole expression in a sub-select. The rule computes the amount of each line in the group pattern, and adds the shipping after the group element:

In [20]:
orders = StarLayerGraph(prefixes={"ex": EX})
orders.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:order1 a ex:Order ; ex:line ex:l1, ex:l2 .
    ex:l1 ex:price 10 ; ex:quantity 2 .
    ex:l2 ex:price 5 ; ex:quantity 3 .

    ex:order2 a ex:Order ; ex:line ex:l3 .
    ex:l3 ex:price 7 ; ex:quantity 1 .
""", format="turtle")

# SPARQL: the aggregate is part of an expression
total_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?o ex:total ?t }
WHERE {
    SELECT ?o (SUM(?p * ?q) + 5 AS ?t)
    WHERE { ?o ex:line ?l . ?l ex:price ?p . ?l ex:quantity ?q }
    GROUP BY ?o
}
"""

# SRL: SET inside the group pattern, and SET after the group element
total_rule = """
PREFIX ex: <http://example.org/>
RULE { ?o ex:total ?t }
WHERE {
    ?o a ex:Order .
    GROUP BY ?o {
        ?o ex:line ?l . ?l ex:price ?p . ?l ex:quantity ?q
        SET(?amount := ?p * ?q)
    } [ ?sum := SUM(?amount) ]
    SET(?t := ?sum + 5)
}
"""

by_construct = orders.query(total_query).graph
by_rule = srl.parse_ruleset(total_rule).infer(orders)
by_rule.bind("ex", EX)
show(by_rule)
print("same triples:", isomorphic(by_construct, by_rule))

@prefix ex: <http://example.org/> .

ex:order1 ex:total 40 .

ex:order2 ex:total 12 .

same triples: True


## 04.04.12 Solution modifiers

SPARQL's solution modifiers shape the solutions of a query: `ORDER BY`, `LIMIT`, `OFFSET`, `DISTINCT`, `REDUCED` and `HAVING`. SRL includes none of them. A rule set adds triples to a graph, and a graph is an unordered set.

**`DISTINCT` and `REDUCED`** remove duplicate rows from the results of a `SELECT`. They have no impact on the creation of triples, so they are not relevant to a SPARQL `CONSTRUCT` or to an SRL rule. A graph holds each triple once, so a rule that creates the same triple twice creates it once.

**`ORDER BY`, `LIMIT` and `OFFSET`.** In a `CONSTRUCT`, `ORDER BY` alone has no effect, because a graph has no order. With `LIMIT` and `OFFSET` it decides which solutions create triples, such as the top N. A rule has no equivalent. Triples have no order, and without an `ORDER BY` the choice would be arbitrary, so the same rules and base graph would no longer always give the same triples. Section 04.04.12.1 shows two ways to get the same results: rank rules, or a query on the inference graph afterwards.

**`HAVING`** filters groups after they have been aggregated. `GROUP BY ?x` collects the solutions of the `WHERE` into one group for each `?x`, and computes aggregates such as `COUNT(?v)` for each group. `HAVING (COUNT(?v) > 1)` then keeps only the groups whose aggregate meets the condition. 

In SRL the group element does the grouping: `GROUP BY ?x { pattern } [ ?n := COUNT(*) ]` stores the aggregate for each `?x` in the variable `?n`. A `FILTER(?n > 1)` written after the group element then keeps the groups that meet the condition. There are two differences from SPARQL:
- The aggregate must be stored in a variable first. It cannot be written inside the `FILTER`, as `HAVING (COUNT(?v) > 1)` writes it.
- A condition on the rows before they are grouped, which SPARQL puts in the `WHERE`, goes in the group pattern as a `FILTER`.


The following code counts the vulnerabilities of each component. The `CONSTRUCT` uses `HAVING` to keep the components with more than one, and the rule uses a `FILTER` after the group element:

In [19]:
findings = StarLayerGraph(prefixes={"ex": EX})
findings.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:app    a ex:Component ; ex:hasVulnerability ex:v1, ex:v2, ex:v3 .
    ex:cache  a ex:Component ; ex:hasVulnerability ex:v4, ex:v5 .
    ex:db     a ex:Component ; ex:hasVulnerability ex:v6 .
    ex:logger a ex:Component .
""", format="turtle")

# SPARQL: HAVING keeps the components with more than one vulnerability
having_query = """
PREFIX ex: <http://example.org/>
CONSTRUCT { ?x ex:vulnerabilityCount ?n }
WHERE {
    SELECT ?x (COUNT(?v) AS ?n)
    WHERE { ?x ex:hasVulnerability ?v }
    GROUP BY ?x
    HAVING (COUNT(?v) > 1)
}
"""

# SRL: a FILTER on the aggregate, after the group element
having_rule = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:vulnerabilityCount ?n }
WHERE {
    ?x a ex:Component .
    GROUP BY ?x { ?x ex:hasVulnerability ?v } [ ?n := COUNT(*) ]
    FILTER(?n > 1)
}
"""

by_construct = findings.query(having_query).graph
by_rule = srl.parse_ruleset(having_rule).infer(findings)
by_rule.bind("ex", EX)
show(by_rule)
print("same triples:", isomorphic(by_construct, by_rule))

@prefix ex: <http://example.org/> .

ex:app ex:vulnerabilityCount 3 .

ex:cache ex:vulnerabilityCount 2 .

same triples: True


### 04.04.12.1 `ORDER BY`, `LIMIT` and rank

In SRL there are two ways to get the effect of `ORDER BY` and `LIMIT`:

- An SRL alternative to `ORDER BY` is to use a `GROUP BY`, `FILTER` and `COUNT` to assign a rank to solution rows.
- An SRL alternative to `LIMIT` is to use the rank to identify selected triples in the inference graph and filter on that using `NOT`.

Note that an often simpler approach is to use SPARQL `ORDER BY` and `LIMIT` to query the inference graph once the SRL rule processing is completed.

The following code ranks and selects the top-three finishers, including ties for the rule-based approach.

In [20]:
scores = StarLayerGraph(prefixes={"ex": EX})
scores.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:ann ex:score 90 .
    ex:eve ex:score 88 .
    ex:cat ex:score 82 .
    ex:fay ex:score 82 .
    ex:bob ex:score 75 .
    ex:dan ex:score 60 .
""", format="turtle")

# SPARQL: ORDER BY and LIMIT keep exactly three
top_three = scores.query("""
    PREFIX ex: <http://example.org/>
    SELECT ?x ?s WHERE { ?x ex:score ?s } ORDER BY DESC(?s) LIMIT 3
""")
print("ORDER BY and LIMIT 3:")
for row in top_three:
    print("   ", row["x"].value.removeprefix(EX), row["s"].value)

# SRL: a rule for the player with nobody above, and a rule for the players with fewer than three above
rank_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:topThree ?s }
WHERE { ?x ex:score ?s . NOT { ?y ex:score ?t FILTER(?t > ?s) } }

RULE { ?x ex:topThree ?s }
WHERE {
    ?x ex:score ?s .
    GROUP BY ?s { ?y ex:score ?t FILTER(?t > ?s) } [ ?above := COUNT(*) ]
    FILTER(?above < 3)
}
"""

ranked = srl.parse_ruleset(rank_rules).infer(scores)
print("rank rules, ex:topThree, including ties:")
for subject, _, score in sorted(ranked, key=lambda triple: -int(triple[2].value)):
    print("   ", subject.value.removeprefix(EX), score.value)

ORDER BY and LIMIT 3:
    ann 90
    eve 88
    fay 82
rank rules, ex:topThree, including ties:
    ann 90
    eve 88
    cat 82
    fay 82


## 04.04.13 SPARQL functions excluded from SRL

- **`RAND`.** The SRL specification leaves it out because it generates different results each time it is called and SRL is designed to produce the same set of triples based on the same rule set and same base graph.

- **The hash functions** `MD5`, `SHA1`, `SHA256`, `SHA384` and `SHA512`. The SRL specification does not support these hash expressions.  

- **Custom functions.** StarLayer SPARQL, which uses Oxigraph, allows custom functions, whether provided by the application or built into the Oxigraph engine (for example its `xsd:date` cast). SRL allows the XSD casts such as `xsd:integer(?x)`, but does not support custom functions. A rule set that calls one is rejected when it is parsed.




## 04.04.14 Summary

What a SPARQL feature becomes in SRL:

- **`CONSTRUCT` and `INSERT`** are a rule (section 04.04.2).
- **Datasets, `GRAPH`, `FROM` and `FROM NAMED`** have no counterpart. A rule runs on one graph, or the union of a dataset (section 04.04.3).
- **`SERVICE`** is not supported. Query the endpoint first (section 04.04.3).
- **`INSERT DATA`** is a `DATA` block (section 04.04.5).
- **`BIND`** is `SET` (section 04.04.6).
- **`|`, `*`, `+`, `?` and `!` in a property path** are rules (section 04.04.7).
- **A `WHERE` that must not read derived triples** is `WHERE GROUND` or `NOT GROUND` (section 04.04.8).
- **`EXISTS`** is a pattern in the body, and **`NOT EXISTS`** is `NOT` (sections 04.04.9 and 04.04.10).
- **`OPTIONAL`, `COALESCE` and `BOUND`** are rules split by what each triple needs, with `NOT` for the case with no match (section 04.04.11.1).
- **`UNION`** is one rule for each branch (section 04.04.11.2).
- **`MINUS`** is `NOT`, when the pattern shares a variable (section 04.04.11.3).
- **`VALUES`** is `DATA` triples matched in the body (section 04.04.11.4).
- **Sub-selects** are rules that create interim triples (section 04.04.11.5).
- **Aggregates in expressions** are `SET` inside the group pattern, or after the group element (section 04.04.11.6).
- **`HAVING`** is a `FILTER` after the group element (section 04.04.12).
- **`ORDER BY`, `LIMIT` and `OFFSET`** are rank rules, or a query on the inference graph (section 04.04.12.1).
- **`DISTINCT` and `REDUCED`** are not needed.
- **`RAND`, the hash functions and custom functions** are not supported (section 04.04.13).